# Обучение модели для распознования тем Уровень 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers scikit-learn tqdm pandas torch

In [ ]:
!pip install -U transformers accelerate
import os; os._exit(00)  # Это сбросит runtime — после этого запускайте всё заново

In [ ]:
    import transformers
    print(transformers.__version__)

4.57.1


In [ ]:
import os
import torch
os.environ["WANDB_DISABLED"] = "true"

DATASET_PATH = "s7_dataset_fixed.csv"
MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"
MODEL_PATH = "./drive/MyDrive/s7_level1_model_tune_tune"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import csv

input_file = './s7_dataset.csv'
output_file = './s7_dataset_fixed.csv'

def fix_csv_file(input_file, output_file):
  rows_fixed = []

  with open(input_file, 'r', encoding='utf-8-sig', newline='') as infile:
    lines = infile.readlines()

    header = ['message_id', 'text', 'level1_category', 'level2_labels']
    rows_fixed.append(header)

    for i, line in enumerate(lines[1:], start=1):
      line = line.strip()
      if not line:
        continue

      if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
        line = line.replace('""', '"')

      first_comma = line.find(',')
      if first_comma == -1:
        print(f"Предупреждение: строка {i} пропущена (нет запятой)")
        continue

      message_id = line[:first_comma].strip()
      rest = line[first_comma+1:]

      parts = rest.rsplit(',', 2)

      if len(parts) == 3:
        text = parts[0].strip()
        level1 = parts[1].strip()
        level2 = parts[2].strip()
        rows_fixed.append([message_id, text, level1, level2])
      else:
        print(f"Предупреждение: строка {i} не разобрана корректно")
        continue

    with open(output_file, 'w', encoding='utf-8', newline='') as outfile:
        writer = csv.writer(
          outfile,
          delimiter=',',
          quotechar='"',
          quoting=csv.QUOTE_MINIMAL
          )

        for row in rows_fixed:
          writer.writerow(row)

    print(f"✓ Файл успешно исправлен!")
    print(f"✓ Исходный файл: {input_file}")
    print(f"✓ Исправленный файл: {output_file}")
    print(f"✓ Обработано строк: {len(rows_fixed)} (включая заголовок)")

    return len(rows_fixed)

fix_csv_file(input_file, output_file)

✓ Файл успешно исправлен!
✓ Исходный файл: ./s7_dataset.csv
✓ Исправленный файл: ./s7_dataset_fixed.csv
✓ Обработано строк: 11906 (включая заголовок)


11906

In [ ]:
import pandas as pd
import numpy as np
import torch
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from tqdm.auto import tqdm
from collections import Counter

print("🔹 Загрузка и подготовка данных...")
df = pd.read_csv(DATASET_PATH)

print(df['level1_category'].isna().sum())
df = df[~df['level1_category'].isna()]
df['level2_labels'] = df['level2_labels'].fillna('Другое')
df['level2_list'] = df['level2_labels'].astype(str).str.split(';')

cat_cleanup = {
    "Лояцность и продажи": "Лояльность и продажи",
    "До Вылетa": "До Вылета"
}
df['level1_category'] = df['level1_category'].replace(cat_cleanup)
print(df['level1_category'].value_counts())

le1 = LabelEncoder()
df['level1_label_idx'] = le1.fit_transform(df['level1_category'])
X = df['text'].astype(str).tolist()
y_series = df['level1_label_idx']

# Контролируем полное распределение классов
print("🔹 Распределение исходных классов:")
for idx, name in enumerate(le1.classes_):
    print(f"{name}: {Counter(y_series)[idx]}")


# Нет фильтрации малых классов!
X_filtered = X
y_filtered = y_series.tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X_filtered, y_filtered, test_size=0.15, random_state=SEED, stratify=y_filtered
)

# Проверяем распределение классов train/val
print("🔹 Классы в train:")
print(Counter(y_train))
print(le1.inverse_transform(sorted(set(y_train))))

print("🔹 Классы в val:")
print(Counter(y_val))
print(le1.inverse_transform(sorted(set(y_val))))

print("🔹 Токенизация данных для Level 1...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_encodings = tokenizer(
    X_train, truncation=True, padding=True, max_length=MAX_LEN
)
val_encodings = tokenizer(
    X_val, truncation=True, padding=True, max_length=MAX_LEN
)

class TextClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextClassificationDataset(train_encodings, y_train)
val_dataset = TextClassificationDataset(val_encodings, y_val)

print("🔹 Вычисление весов классов для loss-функции...")
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

def compute_metrics_level1(p):
    preds = np.argmax(p.predictions, axis=1)
    accuracy = accuracy_score(p.label_ids, preds)
    f1 = f1_score(p.label_ids, preds, average="macro")
    return {"accuracy": accuracy, "f1_macro": f1}

# Кастомная Trainer с учетом взвешивания классов
from transformers import Trainer
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args1 = TrainingArguments(
    output_dir=MODEL_PATH,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    save_strategy="epoch",
    metric_for_best_model="f1_macro",
    save_total_limit=2,
    seed=SEED,
    fp16=True if DEVICE == 'cuda' else False,
    logging_steps=100,
)

model1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le1.classes_)
).to(DEVICE)

trainer1 = WeightedTrainer(
    model=model1,
    args=training_args1,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_level1,
    tokenizer=tokenizer,
)

print("🔹 Обучение модели Level 1...")
trainer1.train()

trainer1.save_model(MODEL_PATH)
joblib.dump(le1, f"{MODEL_PATH}/level1_label_encoder.joblib")
print("✅ Level 1 обучение завершено и модель сохранена.")

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("🔹 Оценка качества на отложенной выборке...")
# Генерируем предсказания модели для текстов из валидационного датасета
preds_output = trainer1.predict(val_dataset)
preds = np.argmax(preds_output.predictions, axis=1)
true = y_val

# Выводим отчёт по классам, включая F1, precision, recall для каждой категории
print(classification_report(true, preds, target_names=le1.inverse_transform(sorted(set(true)))))

# Если нужно вывести confusion matrix
print("Confusion matrix:\n", confusion_matrix(true, preds))

# Отдельно можно посчитать для документации:
from sklearn.metrics import accuracy_score, f1_score
print(f"Accuracy: {accuracy_score(true, preds):.4f}")
print(f"F1 (macro): {f1_score(true, preds, average='macro'):.4f}")
print(f"F1 (weighted): {f1_score(true, preds, average='weighted'):.4f}")

🔹 Загрузка и подготовка данных...
0
level1_category
После вылета            1500
Технические вопросы     1500
Лояльность и продажи    1500
До Вылета               1455
Вылет Прилет            1450
На борту                1450
Обратная связь          1400
Другое                  1200
Сервисные                450
Name: count, dtype: int64
🔹 Распределение исходных классов:
Вылет Прилет: 1450
До Вылета: 1455
Другое: 1200
Лояльность и продажи: 1500
На борту: 1450
Обратная связь: 1400
После вылета: 1500
Сервисные: 450
Технические вопросы: 1500
🔹 Классы в train:
Counter({6: 1275, 3: 1275, 8: 1275, 1: 1237, 0: 1232, 4: 1232, 5: 1190, 2: 1020, 7: 383})
['Вылет Прилет' 'До Вылета' 'Другое' 'Лояльность и продажи' 'На борту'
 'Обратная связь' 'После вылета' 'Сервисные' 'Технические вопросы']
🔹 Классы в val:
Counter({3: 225, 8: 225, 6: 225, 4: 218, 0: 218, 1: 218, 5: 210, 2: 180, 7: 67})
['Вылет Прилет' 'До Вылета' 'Другое' 'Лояльность и продажи' 'На борту'
 'Обратная связь' 'После вылета' 'Сервисн

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


🔹 Вычисление весов классов для loss-функции...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-799468931.py:131: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer1 = WeightedTrainer(


🔹 Обучение модели Level 1...


Step,Training Loss
100,0.975100
200,0.153900
300,0.060800
400,0.059900
500,0.033100
600,0.080300
700,0.034100
800,0.020400
900,0.013500
1000,0.018400


✅ Level 1 обучение завершено и модель сохранена.
🔹 Оценка качества на отложенной выборке...


                      precision    recall  f1-score   support

        Вылет Прилет       1.00      1.00      1.00       218
           До Вылета       1.00      1.00      1.00       218
              Другое       0.99      1.00      0.99       180
Лояльность и продажи       1.00      0.98      0.99       225
            На борту       1.00      1.00      1.00       218
      Обратная связь       1.00      1.00      1.00       210
        После вылета       1.00      1.00      1.00       225
           Сервисные       1.00      0.99      0.99        67
 Технические вопросы       0.99      1.00      0.99       225

            accuracy                           1.00      1786
           macro avg       1.00      0.99      1.00      1786
        weighted avg       1.00      1.00      1.00      1786

Confusion matrix:
 [[217   0   0   0   0   0   0   0   1]
 [  0 218   0   0   0   0   0   0   0]
 [  0   0 180   0   0   0   0   0   0]
 [  0   0   2 220   0   1   0   0   2]
 [  0   0   0   